# INNER JOIN

## Overview
`INNER JOIN` (or just `JOIN`) returns only rows where the join condition is TRUE in **both** tables. Rows that have no match in the other table are excluded from the result.

**Syntax:**
```sql
SELECT  t1.col, t2.col
FROM    table1 AS t1
INNER JOIN table2 AS t2
    ON  t1.key = t2.key
WHERE   ...;
```

**Key concepts:**

| Concept | Detail |
|---|---|
| Join condition | Any boolean expression in `ON` — most commonly an equality on a foreign key |
| Table aliases | `AS t1` (or just `t1`) — essential when column names collide or queries get long |
| Column qualification | `table.column` or `alias.column` — required when both tables share a column name |
| Join order | Logically irrelevant for INNER JOIN — the optimiser chooses the physical order |
| Multiple conditions | `ON t1.id = t2.id AND t1.dept = t2.dept` — compound join keys |
| Non-equi joins | `ON t1.date BETWEEN t2.start AND t2.end` — valid but less common |

**What INNER JOIN excludes:**
- Patients with no encounters
- Encounters with no matching patient (orphaned rows)
- Any row where the join key is NULL (NULL never equals NULL)

---

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE patients (
    patient_id  INTEGER PRIMARY KEY,
    first_name  TEXT,
    last_name   TEXT,
    dob         TEXT,
    province    TEXT,
    active      INTEGER DEFAULT 1
);
CREATE TABLE providers (
    provider_id  INTEGER PRIMARY KEY,
    full_name    TEXT,
    specialty    TEXT,
    province     TEXT
);
CREATE TABLE departments (
    dept_id    INTEGER PRIMARY KEY,
    dept_name  TEXT,
    building   TEXT,
    head_provider_id INTEGER   -- FK to providers (self-join demo too)
);
CREATE TABLE encounters (
    enc_id      INTEGER PRIMARY KEY,
    patient_id  INTEGER REFERENCES patients(patient_id),
    provider_id INTEGER REFERENCES providers(provider_id),
    dept_id     INTEGER REFERENCES departments(dept_id),
    enc_date    TEXT,
    diag_code   TEXT,
    cost_cad    REAL,
    admitted    INTEGER DEFAULT 0
);
CREATE TABLE diagnoses (
    diag_code   TEXT PRIMARY KEY,
    description TEXT,
    category    TEXT
);
CREATE TABLE referrals (
    referral_id     INTEGER PRIMARY KEY,
    from_provider   INTEGER REFERENCES providers(provider_id),
    to_provider     INTEGER REFERENCES providers(provider_id),
    patient_id      INTEGER REFERENCES patients(patient_id),
    referral_date   TEXT,
    reason          TEXT
);

INSERT INTO patients VALUES
  (1,'Aroha','Ngata','1985-03-12','NB',1),
  (2,'Liam','Chen','1972-11-04','NS',1),
  (3,'Fatima','Al-Rashid','1990-07-22','ON',1),
  (4,'James','MacLeod','1955-01-30','NB',0),
  (5,'Sofia','Petrov','2001-09-15','BC',1),
  (6,'Noah','Williams','1968-06-08','AB',1),
  (7,'Mei','Zhang','1995-04-17','ON',1),
  (8,'Ethan','Tremblay','1980-12-01','QC',0),
  (9,'Priya','Nair','1977-08-25','BC',1),
  (10,'Marcus','Okafor','1993-05-19','ON',1),
  (11,'Dana','Leblanc','2000-02-14','NB',1);  -- no encounters yet

INSERT INTO providers VALUES
  (10,'Dr. Sarah MacNeil','Cardiology','NB'),
  (11,'Dr. James Wong','Oncology','BC'),
  (12,'Dr. Fatima Al-Rashid','Family Medicine','ON'),
  (13,'Dr. Ethan Tremblay','Orthopaedics','QC'),
  (14,'Dr. Priya Nair','Emergency Medicine','BC'),
  (15,'Dr. Marcus Okafor','Radiology','ON'),
  (16,'Dr. Unassigned','Neurology','NB');  -- no encounters

INSERT INTO departments VALUES
  (1,'Cardiology','Building A',10),
  (2,'Oncology','Building B',11),
  (3,'Family Medicine','Building C',12),
  (4,'Orthopaedics','Building A',13),
  (5,'Emergency','Building D',14),
  (6,'Radiology','Building B',15),
  (7,'Neurology','Building C',16);

INSERT INTO diagnoses VALUES
  ('J18.9','Pneumonia, unspecified','Respiratory'),
  ('I25.1','Atherosclerotic heart disease','Cardiovascular'),
  ('Z00.0','General adult examination','Preventive'),
  ('M17.1','Primary osteoarthritis, knee','Musculoskeletal'),
  ('J06.9','Acute upper respiratory infection','Respiratory'),
  ('R07.9','Chest pain, unspecified','Cardiovascular'),
  ('I10',  'Essential hypertension','Cardiovascular'),
  ('R55',  'Syncope and collapse','Neurological'),
  ('I48.0','Paroxysmal atrial fibrillation','Cardiovascular'),
  ('S52.5','Fracture of lower end of radius','Musculoskeletal'),
  ('M54.5','Low back pain','Musculoskeletal');

INSERT INTO encounters VALUES
  (1, 1,10,1,'2023-01-15','J18.9',1850.00,1),
  (2, 2,10,1,'2023-02-20','I25.1',3200.00,1),
  (3, 3,12,3,'2023-03-05','Z00.0', 120.00,0),
  (4, 4,13,4,'2023-03-18','M17.1',5500.00,1),
  (5, 5,14,5,'2023-04-02','J06.9',  95.00,0),
  (6, 6,10,1,'2023-05-11','R07.9', 780.00,0),
  (7, 7,10,1,'2023-06-22','I10',  2100.00,1),
  (8, 8,12,3,'2023-07-14',NULL,     80.00,0),
  (9, 1,14,5,'2023-08-30','R55',   410.00,0),
  (10,3,12,3,'2023-09-12','Z00.0', 110.00,0),
  (11,9,10,1,'2023-10-01','I48.0',1750.00,1),
  (12,10,13,4,'2023-11-03','S52.5',2200.00,1),
  (13,2,12,3,'2023-11-18',NULL,     90.00,0),
  (14,5,13,4,'2023-12-05','M54.5', 450.00,0);

INSERT INTO referrals VALUES
  (1,12,10,3,'2023-03-10','Chest pain follow-up'),
  (2,10,11,2,'2023-03-01','Suspected malignancy'),
  (3,14,10,9,'2023-09-05','AF workup'),
  (4,12,13,5,'2023-12-01','Back pain unresponsive to treatment'),
  (5,10,15,6,'2023-06-15','Imaging for chest pain');
""")
conn.commit()
print("Schema ready — tables: patients, providers, departments, encounters, diagnoses, referrals")
print()

# Quick row counts
for t in ['patients','providers','departments','encounters','diagnoses','referrals']:
    n = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'  {t}: {n} rows')


---
## Basic INNER JOIN: encounters with patient names

In [ ]:
# Join encounters to patients — only encounters with a matching patient are returned
print("=== Encounters with patient names (INNER JOIN) ===")
q = """
SELECT  e.enc_id,
        p.first_name || ' ' || p.last_name  AS patient,
        e.enc_date,
        e.diag_code,
        e.cost_cad
FROM    encounters  AS e
INNER JOIN patients AS p
    ON  e.patient_id = p.patient_id
ORDER BY e.enc_date
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Note: patient 11 (Dana Leblanc) has no encounters — she does NOT appear
print()
print("Patient 11 (Dana Leblanc) has no encounters — excluded from INNER JOIN result.")
print(f"encounters: {conn.execute('SELECT COUNT(*) FROM encounters').fetchone()[0]} rows")
print(f"result rows: {len(pd.read_sql(q, conn))} rows")


---
## Joining three tables: encounters + patients + diagnoses

In [ ]:
# Add diagnosis descriptions by joining the diagnoses lookup table
print("=== Encounters with patient + diagnosis description ===")
q = """
SELECT  e.enc_id,
        p.first_name || ' ' || p.last_name  AS patient,
        e.enc_date,
        e.diag_code,
        d.description                        AS diagnosis,
        d.category,
        e.cost_cad,
        CASE e.admitted WHEN 1 THEN 'Yes' ELSE 'No' END AS admitted
FROM    encounters   AS e
INNER JOIN patients  AS p  ON e.patient_id = p.patient_id
INNER JOIN diagnoses AS d  ON e.diag_code  = d.diag_code
ORDER BY e.enc_date
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Encounters 8 and 13 have NULL diag_code — excluded because NULL ON NULL never matches
print()
print("Encounters 8 and 13 have NULL diag_code — excluded from both INNER JOINs.")


---
## Column qualification and aliasing

In [ ]:
# Both patients and providers have a 'province' column
# Must qualify to avoid ambiguity
print("=== Encounters: patient province vs provider province ===")
q = """
SELECT  e.enc_id,
        p.first_name || ' ' || p.last_name   AS patient,
        p.province                            AS patient_province,
        prov.full_name                        AS provider,
        prov.province                         AS provider_province,
        e.cost_cad
FROM    encounters   AS e
INNER JOIN patients  AS p    ON e.patient_id  = p.patient_id
INNER JOIN providers AS prov ON e.provider_id = prov.provider_id
ORDER BY e.enc_id
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Flag cross-province encounters (patient and provider in different provinces)
print()
print("=== Cross-province encounters (patient province != provider province) ===")
q2 = """
SELECT  e.enc_id,
        p.first_name || ' ' || p.last_name  AS patient,
        p.province                           AS pt_prov,
        prov.full_name                       AS provider,
        prov.province                        AS dr_prov
FROM    encounters  AS e
INNER JOIN patients  AS p    ON e.patient_id  = p.patient_id
INNER JOIN providers AS prov ON e.provider_id = prov.provider_id
WHERE   p.province <> prov.province
"""
print(pd.read_sql(q2, conn).to_string(index=False))


---
## Non-equi joins and compound join conditions

In [ ]:
# Non-equi join: find diagnoses in same category as each encounter's diagnosis
print("=== Each encounter matched to all diagnoses in the same ICD category ===")
q = """
SELECT  e.enc_id,
        e.diag_code                 AS enc_diag,
        d_enc.category,
        d_other.diag_code           AS related_diag,
        d_other.description         AS related_description
FROM    encounters   AS e
INNER JOIN diagnoses AS d_enc   ON e.diag_code     = d_enc.diag_code
INNER JOIN diagnoses AS d_other ON d_enc.category  = d_other.category
                                AND d_other.diag_code <> e.diag_code   -- exclude self
ORDER BY e.enc_id, d_other.diag_code
LIMIT 15
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Compound join key: hypothetical multi-column key example
print()
print("=== Compound join key pattern (reference) ===")
print("""
-- When a relationship requires matching on multiple columns:
SELECT ...
FROM   table_a AS a
INNER JOIN table_b AS b
    ON  a.patient_id = b.patient_id
    AND a.visit_year  = b.visit_year   -- second key column
WHERE  ...
""")


---
## Aggregating after a join

In [ ]:
# Count encounters and total cost per provider
print("=== Provider activity summary ===")
q = """
SELECT  prov.full_name                      AS provider,
        prov.specialty,
        COUNT(e.enc_id)                     AS encounters,
        SUM(e.admitted)                     AS admissions,
        ROUND(AVG(e.cost_cad), 2)           AS avg_cost,
        ROUND(SUM(e.cost_cad), 2)           AS total_billed
FROM    providers   AS prov
INNER JOIN encounters AS e ON e.provider_id = prov.provider_id
GROUP BY prov.provider_id, prov.full_name, prov.specialty
ORDER BY total_billed DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Note: providers 11 (Oncology), 16 (Neurology) have no encounters — excluded
print()
print("Providers 11 (Oncology) and 16 (Neurology) have no encounters — excluded.")
print("Use LEFT JOIN if you want all providers including those with zero encounters.")

conn.close()


---
## Common Pitfalls

**1. INNER JOIN silently excludes non-matching rows**
If a patient has no encounters, or an encounter has a NULL foreign key, they are excluded from the result with no error or warning. Always check row counts before and after a join to verify you're not silently losing data you expected to keep.

**2. NULL foreign keys never match**
`NULL = NULL` evaluates to NULL, not TRUE. An encounter with `patient_id = NULL` is excluded from every INNER JOIN on `patient_id`, even if the patients table had a NULL primary key. Enforce NOT NULL constraints on foreign keys and check for NULLs before joining.

**3. Omitting table aliases on shared column names causes errors or wrong results**
Both `patients` and `providers` have a `province` column. Writing `SELECT province` without a table qualifier either raises an ambiguity error (PostgreSQL) or silently picks one (SQLite). Always qualify column names: `p.province` vs `prov.province`.

**4. Forgetting to qualify columns leads to ambiguous GROUP BY**
`GROUP BY provider_id` is ambiguous if both joined tables have a `provider_id` column. Use the alias: `GROUP BY prov.provider_id`. In PostgreSQL this raises an error; in SQLite it may silently choose one.

**5. Cartesian product from a missing ON clause**
`FROM encounters, patients` without a WHERE/ON condition produces a cartesian product — every encounter row paired with every patient row. In modern SQL, always use explicit `JOIN ... ON` syntax. Implicit comma joins are a legacy pattern and a common source of accidental fan-out.

**6. Fan-out when joining to a one-to-many relationship**
If an encounter can have multiple diagnosis codes (one-to-many), joining on `diag_code` will duplicate encounter rows — one per matching diagnosis. This inflates SUM() and COUNT() aggregates. Always check the cardinality of your join relationship before aggregating.


---
*sql_methods_library - Samantha McGarrigle*